# Quantum Volume

Quantum volume is a benchmark for the effective performance of a quantum processor, capturing the impact of errors across multiqubit circuits. It is based on random circuits and the **heavy output probability**, namely the probability of measuring bitstrings whose ideal probabilities are above the median of the noiseless output distribution. For an ideal device this probability is approximately $0.85$. A width is considered successful if the measured heavy output probability exceeds $2/3$ with a $2\sigma$ confidence margin. The quantum volume is then defined as $\mathrm{QV}=2^n$, where $n$ is the largest width that passes.

#### The model
For $n$ qubits, the model consists of $n$ layers. In each layer, the qubits are randomly paired, and a Haar-random SU(4) gate is applied to each pair. After all layers, the circuit is measured. The heavy outputs are determined from the noiseless distribution of the same random circuit, which is why the protocol is mainly practical for moderate-size QPUs.

#### The protocol
For a fixed width $n$, we generate $T$ independent random circuits from this ensemble and execute them on the device. For each circuit, we estimate the probability of obtaining a heavy output from the measurement data, and then average this quantity over the $T$ trials. The width $n$ passes if this average is greater than $2/3$ with the required $2\sigma$ statistical confidence. The reported quantum volume is $\mathrm{QV}=2^n$, where $n$ is the largest successful width.

In [1]:
import sys

sys.path.insert(0, "..")

from hardware import HardwareRunner
from hardwares_preferences import HARDWARES

The quantum model is already defined in the `QuantumVolumeProtocol` class. We first define our hardware runners

In [2]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}


classiq_runners = [
    HardwareRunner("Classiq", backend_name, **common_config)
    for backend_name in HARDWARES["Classiq"][0:2]
]

ionq_runners = [
    HardwareRunner("IonQ", backend_name, **common_config)
    for backend_name in HARDWARES["IonQ"][0:1]
]


ibm_runners = [
    HardwareRunner(
        "IBM Quantum",
        backend_name,
        **common_config,
        backend_kwargs={
            "access_token": "PUT YOUR TOKEN HERE",
            "channel": "PUT YOUR CHANNEL HERE",
            "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
        },
    )
    for backend_name in HARDWARES["IBM Quantum"][0:1]
]

runners = [
    *classiq_runners,
    *ionq_runners,
    *ibm_runners,
]

Next, we define a `QuantumVolumeProtocol` with the minimal and maximal number of qubits, and the number of trails (a suggested number is around 100, here we take only 10). 

In [3]:
from protocol import QuantumVolumeProtocol

protocol = QuantumVolumeProtocol(
    min_num_qubits=2,
    max_num_qubits=4,
    num_trials=10,
    runners=runners,
    results_dir="results/quantum_volume",
    max_submitted_jobs_in_dir=8,
)

We run the protocol. What happens in practice is that for a given number of qubits, all jobs are sent asynchronously. For convenience we define a loop with some waiting time. 

In [ ]:
import asyncio

from IPython.display import clear_output, display

await protocol.run()

# while True:
#     await protocol.run()

#     summary = protocol.all_width_summaries()

#     clear_output(wait=True)

#     if not summary.empty and (summary["num_completed"] == summary["num_trials_requested"]).all():
#         print("All trials completed.")
#         break

#     await asyncio.sleep(10)

2026-03-18 22:03:46.062888: Submit qv_3_3-3 for IBM Quantum - ibm_kingston


Finally, we can print the summary of the results, as well as update the report.

In [ ]:
protocol.all_width_summaries()
display(protocol.quantum_volume_summary())
display(protocol.final_summary())
display(await protocol.update_report(build=True))